# Partition B+C · Development experiments
Becca’s experiment notebook for Hayden’s frozen case release. The outcome is the current appeal: 0 = affirmed, 1 = reversed/vacated. Run the cells in order after the release arrives. All reported scores use the same 2021–2022 development cases. The 2023–2024 period is reserved.

## 1. Select the release and run directory
The defaults assume `inputs/bc_v1/release.json` next to this notebook. Set `BC_RELEASE` and `BC_RUN` to use other paths. Each run needs a fresh output directory. The full experiment includes LEGAL-BERT. Set `RUN_ENCODER = False` for a lexical-only check under its own run name.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

candidates = [Path.cwd(), Path.cwd() / "experiments/provisional_b"]
HERE = next(p.resolve() for p in candidates if (p / "provisional_b.ipynb").is_file())
sys.path.insert(0, str(HERE))
from experiment import prepare, save_run
from modeling import BERT, LEXICAL, SEED, evaluate
from encoder import feature_vectors

RELEASE = Path(os.environ.get("BC_RELEASE", HERE / "inputs/bc_v1/release.json"))
OUTPUT = Path(os.environ.get("BC_RUN", HERE / "runs/bc_v1"))
RUN_ENCODER = True
if OUTPUT.exists():
    raise FileExistsError(f"Choose a fresh run directory: {OUTPUT}")
release, rows, folds = prepare(RELEASE)
print("Dataset", release["dataset_version"], "· split", release["split_version"],
      "· mask", release["mask_version"], "· CV folds", len(folds))
display(pd.crosstab([pd.Series([r["period"] for r in rows], name="Period"),
                     pd.Series([r["batch"] for r in rows], name="Batch")],
                    pd.Series([r["label"] for r in rows], name="Outcome")))

## 2. Establish the training-majority score
Learn the most frequent label from the 2010–2020 training cases. Report it on the same development IDs used below. FULL and MASKED produce identical predictions for this classifier.

In [ ]:
majority = evaluate("Majority", rows, folds)
display(pd.DataFrame(majority["metrics"]))

## 3. Select the lexical models on MASKED training folds
The primary baseline is TF-IDF with logistic regression. The linear SVM uses the same unigram/bigram features. Each training fold fits its vocabulary, IDF, balanced class weights and classifier. Folds keep families together and contain both outcomes. Three folds are preferred, with two used when the number of families requires it.

Select C from 0.1, 1 and 10 by mean balanced accuracy across folds, then macro F1, then smaller C. Refit FULL and MASKED separately on all training cases using that model’s selected C.

In [ ]:
lexical = [evaluate(name, rows, folds) for name in LEXICAL]

def selection_table(models):
    return pd.DataFrame([{"model": r["model"], "C": t["C"],
                          "mean_fold_balanced_accuracy": t["balanced_accuracy"],
                          "mean_fold_macro_f1": t["macro_f1"], "selected": t["C"] == r["selected_C"]}
                         for r in models for t in r["cv"]])

display(selection_table(lexical))

## 4. Fit logistic regression on frozen LEGAL-BERT vectors
Use every contiguous window of up to 510 content tokens. Average content-token vectors across the whole opinion, excluding specials and padding. Cache each opinion vector. Only the scaler and classifier learn from the training folds. Use the same fold assignments and selection rule as the lexical models.

In [ ]:
models = [majority, *lexical]
if RUN_ENCODER:
    vectors = feature_vectors(rows, HERE / ".cache/vectors", SEED)
    bert = evaluate(BERT, rows, folds, vectors)
    models.append(bert)
    display(selection_table([bert]))

## 5. Report development results
Each row compares the predictions with the reviewed outcomes for the same cases. Confusion matrices use true outcomes as rows and predicted outcomes as columns, ordered affirmed then reversed/vacated. The figure shows counts and within-row percentages.

In [ ]:
directory = save_run(OUTPUT, release, rows, folds, models)
results = pd.read_csv(directory / "results.csv")
display(results)
display(Image(filename=str(directory / "confusion_matrices.png")))
print("Reproducibility files:", directory)

## 6. Interpret the paired change
The difference below is FULL minus MASKED balanced accuracy on the same decisions. A positive value means the FULL model matched more of the two classes on average. Review case counts, class-specific recall and removed passages before attributing a difference to explicit outcome language. Add the reviewed interpretation when reporting this run.

In [ ]:
paired = results.pivot(index="model", columns="variant", values="balanced_accuracy")
paired["FULL minus MASKED"] = paired["FULL"] - paired["MASKED"]
display(paired)